# 02 - Create Quarantine Tables

Creates one Delta quarantine table per required source. Records can be
rejected at either the Bronze parsing stage or the Silver validation
stage while preserving their raw payload and lineage.

In [ ]:
import re

dbutils.widgets.text("catalog", "abc_retail", "Target catalog")
catalog = dbutils.widgets.get("catalog").strip()

if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalog):
    raise ValueError(f"Invalid catalog identifier: {catalog!r}")

spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`quarantine`")

In [ ]:
quarantine_tables = {
    "clickstream_errors": "Rejected clickstream records",
    "orders_errors": "Rejected order records",
    "customers_errors": "Rejected customer records",
}

for table_name, table_comment in quarantine_tables.items():
    spark.sql(
        f"""
        CREATE TABLE IF NOT EXISTS `{catalog}`.`quarantine`.`{table_name}` (
            error_id STRING NOT NULL,
            error_stage STRING NOT NULL,
            error_code STRING NOT NULL,
            error_reason STRING NOT NULL,
            record_key STRING,
            raw_payload STRING,
            rescued_data STRING,
            source_file STRING NOT NULL,
            batch_id STRING,
            load_date DATE,
            ingest_ts TIMESTAMP NOT NULL,
            quarantine_date DATE NOT NULL,
            pipeline_run_id STRING NOT NULL
        )
        USING DELTA
        PARTITIONED BY (quarantine_date)
        TBLPROPERTIES (
            'comment' = '{table_comment}'
        )
        """
    )

In [ ]:
for table_name in quarantine_tables:
    print(f"Created or verified: {catalog}.quarantine.{table_name}")